<a href="https://colab.research.google.com/github/Muqeet-Salam/DVA-Assignment-II/blob/main/Copy_of_dva.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reddit Comment Analysis Pipeline
This notebook runs three analyses on Reddit comment data:
1. **Sentiment Analysis** (VADER)
2. **TF-IDF Vectorization + Clustering** (PCA, t-SNE, K-Means)
3. **Network Analysis** (Author & Thread graphs)
https://chatgpt.com/s/t_69d4d1ecb39c819196583a68f2cad001
**Requirements:** T4 GPU runtime, `reddit_100k.jsonl` in Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#@title 1. Verify T4 GPU
import subprocess, sys

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

if 'T4' in result.stdout:
    print('\n\u2705 T4 GPU detected!')
else:
    print('\n\u26a0\ufe0f T4 GPU NOT detected. Go to Runtime > Change runtime type > T4 GPU')
    sys.exit(1)

In [ ]:
#@title 2. Install Dependencies
!pip install -q vaderSentiment pandas matplotlib seaborn scipy tqdm scikit-learn networkx pyvis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 35.2 MB/s eta 0:00:00


In [ ]:
#@title 3. Mount Google Drive & Set Paths
from google.colab import drive
drive.mount('/content/drive')

import os

# === EDIT THIS: path to your reddit_100k.jsonl inside Google Drive ===
DRIVE_DATA = '/content/drive/MyDrive/reddit.jsonl'

# Working directories
DATA_PATH = DRIVE_DATA
BASE_OUT  = '/content/drive/MyDrive/outputs2'
SENT_OUT  = os.path.join(BASE_OUT, 'sentiment')
CLUST_OUT = os.path.join(BASE_OUT, 'clustering')
NET_OUT   = os.path.join(BASE_OUT, 'network')

for d in [SENT_OUT, CLUST_OUT, NET_OUT]:
    os.makedirs(d, exist_ok=True)

assert os.path.isfile(DATA_PATH), f'Data file not found at {DATA_PATH}\nUpload reddit_100k.jsonl to your Drive and update DRIVE_DATA above.'
print(f'\u2705 Data file found: {DATA_PATH}  ({os.path.getsize(DATA_PATH)/1e6:.1f} MB)')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Data file found: /content/drive/MyDrive/reddit.jsonl  (167.0 MB)


---
## Part 1 — Sentiment Analysis (VADER)

In [ ]:
import json, warnings, pandas as pd, numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from tqdm import tqdm
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
warnings.filterwarnings('ignore')

OUTPUT_DIR = SENT_OUT
SCORE_CAP = 100_000

print('Loading data \u2026')
records = []
with open(DATA_PATH, 'r', encoding='utf-8') as fh:
    for line in tqdm(fh, desc='Reading'):
        try: records.append(json.loads(line))
        except json.JSONDecodeError: continue

df = pd.DataFrame(records)
df['score'] = pd.to_numeric(df['score'], errors='coerce').fillna(0)
df['body'] = df['body'].astype(str)
print(f'Loaded {len(df):,} comments')

print('Running VADER sentiment analysis \u2026')
analyzer = SentimentIntensityAnalyzer()
def get_scores(text):
    return pd.Series(analyzer.polarity_scores(text))
tqdm.pandas(desc='VADER')
df[['compound','neg','neu','pos']] = df['body'].progress_apply(get_scores)

def label(c):
    if c >= 0.05: return 'Positive'
    if c <= -0.05: return 'Negative'
    return 'Neutral'
df['sentiment'] = df['compound'].apply(label)
print(df['sentiment'].value_counts().to_string())

# --- Plots ---
plt.figure(figsize=(10,4))
plt.hist(df['compound'], bins=100, color='#4C72B0', edgecolor='none', alpha=0.85)
plt.axvline(0, color='crimson', linestyle='--', linewidth=1.2)
plt.title('VADER Compound Sentiment Score Distribution', fontsize=14)
plt.xlabel('Compound Score'); plt.ylabel('Number of Comments'); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'1_compound_distribution.png'), dpi=150); plt.show(); plt.close()

counts = df['sentiment'].value_counts()
plt.figure(figsize=(6,6))
plt.pie(counts, labels=counts.index, autopct='%1.1f%%', colors=['#2ecc71','#e74c3c','#95a5a6'],
        startangle=140, wedgeprops=dict(edgecolor='white', linewidth=1.5))
plt.title('Sentiment Label Distribution', fontsize=14); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'2_sentiment_pie.png'), dpi=150); plt.show(); plt.close()

df_plot = df[df['score'].between(-500, SCORE_CAP)].copy()
sample = df_plot.sample(min(5000, len(df_plot)), random_state=42)
slope, intercept, r, p, se = stats.linregress(sample['compound'], sample['score'])
print(f'\nLinear regression compound->score: R={r:.4f}, p={p:.4e}')
fig, ax = plt.subplots(figsize=(9,5))
sc = ax.scatter(sample['compound'], sample['score'], c=sample['compound'], cmap='RdYlGn', alpha=0.3, s=8, rasterized=True)
xs = np.linspace(-1,1,200); ax.plot(xs, slope*xs+intercept, color='black', linewidth=1.5, label=f'OLS R={r:.3f} p={p:.2e}')
plt.colorbar(sc, ax=ax, label='Compound Score')
ax.set_xlabel('Compound Sentiment Score'); ax.set_ylabel('Comment Score (upvotes)'); ax.set_title('Sentiment vs Comment Score', fontsize=14); ax.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'3_sentiment_vs_score.png'), dpi=150); plt.show(); plt.close()

df_box = df[df['score'].between(-200, 5000)]
plt.figure(figsize=(8,5))
sns.boxplot(data=df_box, x='sentiment', y='score', order=['Positive','Neutral','Negative'],
            palette={'Positive':'#2ecc71','Neutral':'#95a5a6','Negative':'#e74c3c'})
plt.title('Comment Score Distribution by Sentiment', fontsize=14); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'4_score_by_sentiment_box.png'), dpi=150); plt.show(); plt.close()

mean_scores = df.groupby('sentiment')['score'].mean().reindex(['Positive','Neutral','Negative'])
plt.figure(figsize=(7,4))
bars = plt.bar(mean_scores.index, mean_scores.values, color=['#2ecc71','#95a5a6','#e74c3c'], edgecolor='white')
for bar, val in zip(bars, mean_scores.values):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'{val:.1f}', ha='center', fontsize=11)
plt.title('Mean Comment Score by Sentiment', fontsize=14); plt.ylabel('Mean Score'); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'5_mean_score_by_sentiment.png'), dpi=150); plt.show(); plt.close()

thread_sent = df.groupby('thread_id').agg(mean_compound=('compound','mean'), count=('compound','count')).query('count >= 20').sort_values('mean_compound')
fig, axes = plt.subplots(1,2, figsize=(14,5))
for ax, title, data in [(axes[0],'10 Most Negative Threads (>=20 comments)', thread_sent.head(10)), (axes[1],'10 Most Positive Threads (>=20 comments)', thread_sent.tail(10))]:
    colors = ['#e74c3c']*10 if 'Negative' in title else ['#2ecc71']*10
    ax.barh(data.index, data['mean_compound'], color=colors); ax.set_xlabel('Mean Compound Score'); ax.set_title(title, fontsize=11)
    ax.set_yticklabels([t[-8:] for t in data.index], fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,'6_thread_sentiment.png'), dpi=150); plt.show(); plt.close()

df['body_len'] = df['body'].str.len()
bins = [0,50,150,300,600,1200,df['body_len'].max()+1]
labels_len = ['<50','50-150','150-300','300-600','600-1200','1200+']
df['len_bin'] = pd.cut(df['body_len'], bins=bins, labels=labels_len)
len_sent = df.groupby('len_bin', observed=True)['compound'].mean()
plt.figure(figsize=(8,4))
plt.bar(len_sent.index, len_sent.values, color='#4C72B0', edgecolor='white')
plt.axhline(0, color='crimson', linestyle='--', linewidth=1)
plt.title('Mean Sentiment by Comment Length', fontsize=14); plt.xlabel('Comment Length (characters)'); plt.ylabel('Mean Compound Score'); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'7_sentiment_by_length.png'), dpi=150); plt.show(); plt.close()

out_csv = os.path.join(OUTPUT_DIR, 'comments_with_sentiment.csv')
df.to_csv(out_csv, index=False)
print(f'\n\u2705 Sentiment analysis complete! Outputs in: {OUTPUT_DIR}')

Loading data …


Reading: 369292it [00:07, 51805.73it/s]


Loaded 369,292 comments
Running VADER sentiment analysis …


VADER: 100%|██████████| 369292/369292 [03:34<00:00, 1721.54it/s]


sentiment
Neutral     199232
Positive    170060

Linear regression compound->score: R=-0.0032, p=8.2349e-01

✅ Sentiment analysis complete! Outputs in: /content/drive/MyDrive/outputs2/sentiment


---
## Part 2 — TF-IDF Vectorization + Clustering

In [ ]:
import json, warnings, numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, matplotlib.colors as mcolors, seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.cluster import MiniBatchKMeans
from tqdm import tqdm
warnings.filterwarnings('ignore')

OUTPUT_DIR = CLUST_OUT
TSNE_SAMPLE = 15000; N_CLUSTERS = 8; TF_MAX_FEAT = 20000; PCA_COMPONENTS = 50; TSNE_PERP = 40; RANDOM_STATE = 42

print('Loading data \u2026')
records = []
with open(DATA_PATH, 'r', encoding='utf-8') as fh:
    for line in tqdm(fh, desc='Reading'):
        try: records.append(json.loads(line))
        except json.JSONDecodeError: continue
df = pd.DataFrame(records)
df['score'] = pd.to_numeric(df['score'], errors='coerce').fillna(0)
df['body'] = df['body'].astype(str)
print(f'Loaded {len(df):,} comments')

print(f'\nBuilding TF-IDF matrix (max_features={TF_MAX_FEAT}) \u2026')
tfidf = TfidfVectorizer(max_features=TF_MAX_FEAT, stop_words='english', sublinear_tf=True, min_df=3, ngram_range=(1,2))
X_tfidf = tfidf.fit_transform(df['body'])
print(f'TF-IDF matrix shape: {X_tfidf.shape}')

print(f'\nRunning MiniBatch K-Means (k={N_CLUSTERS}) \u2026')
km = MiniBatchKMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, batch_size=4096, n_init=5)
df['cluster'] = km.fit_predict(X_tfidf)

print('\n-- Top terms per cluster --')
feature_names = np.array(tfidf.get_feature_names_out())
centers = km.cluster_centers_; cluster_terms = {}
for ci in range(N_CLUSTERS):
    top_idx = centers[ci].argsort()[-10:][::-1]
    terms = ', '.join(feature_names[top_idx]); cluster_terms[ci] = terms
    print(f'  Cluster {ci}: {terms}')

print('\nComputing elbow curve (k=2..12) \u2026')
inertias = []; ks = range(2,13)
for k in tqdm(ks, desc='K-Means'):
    m = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE, batch_size=4096, n_init=3); m.fit(X_tfidf); inertias.append(m.inertia_)

plt.figure(figsize=(8,4)); plt.plot(list(ks), inertias, 'o--', color='#4C72B0')
plt.axvline(N_CLUSTERS, color='crimson', linestyle=':', label=f'Chosen k={N_CLUSTERS}')
plt.title('K-Means Elbow Curve', fontsize=14); plt.xlabel('Number of Clusters (k)'); plt.ylabel('Inertia'); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'1_elbow_curve.png'), dpi=150); plt.show(); plt.close()

cluster_sizes = df['cluster'].value_counts().sort_index()
plt.figure(figsize=(9,4)); plt.bar(cluster_sizes.index, cluster_sizes.values, color=plt.cm.tab10.colors[:N_CLUSTERS])
plt.title('Comment Count per Cluster', fontsize=14); plt.xlabel('Cluster ID'); plt.ylabel('Number of Comments')
for i,v in zip(cluster_sizes.index, cluster_sizes.values): plt.text(i, v+100, str(v), ha='center', fontsize=9)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,'2_cluster_sizes.png'), dpi=150); plt.show(); plt.close()

print('\nRunning TruncatedSVD \u2026')
svd = TruncatedSVD(n_components=PCA_COMPONENTS, random_state=RANDOM_STATE)
if TSNE_SAMPLE and len(df) > TSNE_SAMPLE:
    idx = np.random.RandomState(RANDOM_STATE).choice(len(df), TSNE_SAMPLE, replace=False)
    X_svd_full = svd.fit_transform(X_tfidf); X_svd_sample = X_svd_full[idx]; df_sample = df.iloc[idx].reset_index(drop=True)
else:
    X_svd_full = svd.fit_transform(X_tfidf); X_svd_sample = X_svd_full; df_sample = df.reset_index(drop=True)

var_exp = svd.explained_variance_ratio_
plt.figure(figsize=(9,4)); plt.plot(range(1,PCA_COMPONENTS+1), np.cumsum(var_exp)*100, color='#4C72B0')
plt.fill_between(range(1,PCA_COMPONENTS+1), np.cumsum(var_exp)*100, alpha=0.2, color='#4C72B0')
plt.title('Cumulative Explained Variance (TruncatedSVD/LSA)', fontsize=14); plt.xlabel('Components'); plt.ylabel('Cum. Var. (%)'); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'3_pca_variance.png'), dpi=150); plt.show(); plt.close()

plt.figure(figsize=(9,6))
for ci in range(N_CLUSTERS):
    mask = df_sample['cluster']==ci; plt.scatter(X_svd_sample[mask,0], X_svd_sample[mask,1], s=4, alpha=0.4, label=f'C{ci}', rasterized=True)
plt.title('PCA (SVD) 2-D Projection by Cluster', fontsize=13); plt.xlabel('Component 1'); plt.ylabel('Component 2'); plt.legend(markerscale=4, ncol=2, fontsize=9); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'4_pca_scatter.png'), dpi=150); plt.show(); plt.close()

print(f'\nRunning t-SNE on {len(X_svd_sample):,} samples (perplexity={TSNE_PERP}) \u2026')
tsne = TSNE(n_components=2, perplexity=TSNE_PERP, learning_rate='auto', init='pca', random_state=RANDOM_STATE, n_iter=1000, n_jobs=-1)
X_tsne = tsne.fit_transform(X_svd_sample)
print('t-SNE done.')

cmap = plt.cm.get_cmap('tab10', N_CLUSTERS)
fig, ax = plt.subplots(figsize=(11,8))
for ci in range(N_CLUSTERS):
    mask = df_sample['cluster']==ci; ax.scatter(X_tsne[mask,0], X_tsne[mask,1], color=cmap(ci), s=5, alpha=0.5, label=f'C{ci}', rasterized=True)
ax.set_title('t-SNE Visualization by Cluster', fontsize=14); ax.set_xlabel('t-SNE Dim 1'); ax.set_ylabel('t-SNE Dim 2')
ax.legend(markerscale=4, ncol=2, fontsize=9, title='Cluster'); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'5_tsne_clusters.png'), dpi=150); plt.show(); plt.close()

scores_s = df_sample['score'].clip(lower=1).values; log_sc = np.log1p(scores_s)
fig, ax = plt.subplots(figsize=(11,8))
sc = ax.scatter(X_tsne[:,0], X_tsne[:,1], c=log_sc, cmap='plasma', s=5, alpha=0.5, rasterized=True)
plt.colorbar(sc, ax=ax, label='log(1+score)'); ax.set_title('t-SNE Coloured by Comment Score', fontsize=14); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'6_tsne_score.png'), dpi=150); plt.show(); plt.close()

fig, axes = plt.subplots(2, (N_CLUSTERS+1)//2, figsize=(16,8)); axes = axes.flatten()
for ci in range(N_CLUSTERS):
    scores_c = df[df['cluster']==ci]['score'].clip(-500,5000); axes[ci].hist(scores_c, bins=60, color=cmap(ci), edgecolor='none', alpha=0.8)
    axes[ci].set_title(f'Cluster {ci}', fontsize=10); axes[ci].set_xlabel('Score'); axes[ci].set_ylabel('Count')
for ci in range(N_CLUSTERS, len(axes)): axes[ci].set_visible(False)
plt.suptitle('Score Distribution per Cluster', fontsize=14, y=1.01); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'7_score_per_cluster.png'), dpi=150); plt.show(); plt.close()

with open(os.path.join(OUTPUT_DIR,'cluster_top_terms.json'), 'w') as f:
    json.dump(cluster_terms, f, indent=2)
print(f'\n\u2705 Clustering complete! Outputs in: {OUTPUT_DIR}')

Loading data …


Reading: 369292it [00:02, 159840.41it/s]


Loaded 369,292 comments

Building TF-IDF matrix (max_features=20000) …
TF-IDF matrix shape: (369292, 20000)

Running MiniBatch K-Means (k=8) …

-- Top terms per cluster --
  Cluster 0: hot, don touch, iron, touch, don, sterile, thermostat, fucking, fucking shit, fucking stupid
  Cluster 1: mayor, help think, pet, chicken, asks, help, think, time, fucking sucks, fucking way
  Cluster 2: smoke weed, alcohol, weed, smoke, illegal, dumb, safe, worst people, thing time, od
  Cluster 3: factual, completely forgot, sleeves, troll, elbow, shorts, good friend, scars, wore, assumed
  Cluster 4: affair, quite time, time worked, guy, carrying, groom, married, turns, quite, worked
  Cluster 5: like, just, people, don, time, know, think, good, got, ve
  Cluster 6: surreal, got away, bruh, fuck, away, way, did, got, fucking stupid, fucking sucks
  Cluster 7: valley, stardew valley, stardew, playing, fucking love, fucking stupid, fucking sucks, fucking time, fucking way, fucking weird

Computing elbow

K-Means: 100%|██████████| 11/11 [00:04<00:00,  2.37it/s]



Running TruncatedSVD …

Running t-SNE on 15,000 samples (perplexity=40) …
t-SNE done.

✅ Clustering complete! Outputs in: /content/drive/MyDrive/outputs2/clustering


---
## Part 3 — Network Analysis

In [ ]:
import json, warnings, pandas as pd, numpy as np, networkx as nx
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, matplotlib.cm as cm
from collections import defaultdict
from tqdm import tqdm
import itertools
warnings.filterwarnings('ignore')

OUTPUT_DIR = NET_OUT
TOP_AUTHORS = 300; TOP_THREADS = 200; MIN_COAPPEAR = 3; MAX_EDGE_SAMPLE = 50000

print('Loading data \u2026')
records = []
with open(DATA_PATH, 'r', encoding='utf-8') as fh:
    for line in tqdm(fh, desc='Reading'):
        try: records.append(json.loads(line))
        except json.JSONDecodeError: continue
df = pd.DataFrame(records)
df['score'] = pd.to_numeric(df['score'], errors='coerce').fillna(0)
df['author'] = df['author'].astype(str); df['thread_id'] = df['thread_id'].astype(str)
df = df[~df['author'].isin(['[deleted]','AutoModerator'])]
print(f'Loaded {len(df):,} comments from {df["author"].nunique():,} authors in {df["thread_id"].nunique():,} threads')

author_stats = df.groupby('author').agg(comment_count=('id','count'), total_score=('score','sum'), mean_score=('score','mean'), threads_active=('thread_id','nunique')).sort_values('comment_count', ascending=False)
print(f'\nTop 15 most active authors:\n{author_stats.head(15).to_string()}')

top30 = author_stats.head(30)
plt.figure(figsize=(12,5)); plt.barh(top30.index[::-1], top30['comment_count'][::-1], color='#4C72B0')
plt.xlabel('Number of Comments'); plt.title('Top 30 Most Active Authors', fontsize=14); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'1_top_authors_count.png'), dpi=150); plt.show(); plt.close()

top30s = author_stats.sort_values('total_score', ascending=False).head(30)
plt.figure(figsize=(12,5)); plt.barh(top30s.index[::-1], top30s['total_score'][::-1], color='#2ecc71')
plt.xlabel('Total Karma Score'); plt.title('Top 30 Authors by Total Score', fontsize=14); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'2_top_authors_score.png'), dpi=150); plt.show(); plt.close()

thread_stats = df.groupby('thread_id').agg(comment_count=('id','count'), unique_authors=('author','nunique'), avg_score=('score','mean')).sort_values('comment_count', ascending=False)
plt.figure(figsize=(10,4)); plt.hist(thread_stats['comment_count'], bins=80, color='#e67e22', edgecolor='none', alpha=0.8)
plt.title('Thread Size Distribution', fontsize=14); plt.xlabel('Number of Comments'); plt.ylabel('Number of Threads'); plt.yscale('log'); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'3_thread_size_distribution.png'), dpi=150); plt.show(); plt.close()

print('\nBuilding author co-activity network \u2026')
top_author_set = set(author_stats.head(TOP_AUTHORS).index); top_thread_set = set(thread_stats.head(TOP_THREADS).index)
df_sub = df[df['author'].isin(top_author_set) & df['thread_id'].isin(top_thread_set)]
thread_authors = defaultdict(set)
for _, row in df_sub.iterrows(): thread_authors[row['thread_id']].add(row['author'])
edge_weights = defaultdict(int)
for thread, authors in tqdm(thread_authors.items(), desc='Building edges'):
    for a, b in itertools.combinations(sorted(authors), 2): edge_weights[(a,b)] += 1
print(f'Raw edges: {len(edge_weights):,} -- filtering by min_coappear={MIN_COAPPEAR}')
filtered = {k:v for k,v in edge_weights.items() if v >= MIN_COAPPEAR}
print(f'Edges after filter: {len(filtered):,}')

G = nx.Graph()
for author in top_author_set:
    if author in author_stats.index:
        st = author_stats.loc[author]; G.add_node(author, comment_count=int(st['comment_count']), total_score=float(st['total_score']), threads_active=int(st['threads_active']))
for (a,b), w in list(filtered.items())[:MAX_EDGE_SAMPLE]: G.add_edge(a, b, weight=w)
G.remove_nodes_from(list(nx.isolates(G)))
print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

print('Computing centrality metrics \u2026')
degree_cent = nx.degree_centrality(G); betw_cent = nx.betweenness_centrality(G, normalized=True, k=min(100, G.number_of_nodes()))
nx.set_node_attributes(G, degree_cent, 'degree_centrality'); nx.set_node_attributes(G, betw_cent, 'betweenness_centrality')
top_degree = sorted(degree_cent.items(), key=lambda x:x[1], reverse=True)[:20]
top_between = sorted(betw_cent.items(), key=lambda x:x[1], reverse=True)[:20]
print('\nTop 10 by Degree Centrality:')
for n,v in top_degree[:10]: print(f'  {n:30s}  {v:.4f}')
print('\nTop 10 by Betweenness Centrality:')
for n,v in top_between[:10]: print(f'  {n:30s}  {v:.4f}')

cent_df = pd.DataFrame({'author':list(degree_cent.keys()), 'degree_centrality':list(degree_cent.values()), 'betweenness_centrality':[betw_cent.get(a,0) for a in degree_cent.keys()]}).sort_values('degree_centrality', ascending=False)
cent_df.to_csv(os.path.join(OUTPUT_DIR,'author_centrality.csv'), index=False)

print('Drawing static network \u2026')
draw_G = G.copy()
if draw_G.number_of_nodes() > 150:
    top_nodes = [n for n,_ in sorted(draw_G.degree(), key=lambda x:x[1], reverse=True)[:150]]; draw_G = draw_G.subgraph(top_nodes).copy()
pos = nx.spring_layout(draw_G, seed=42, k=0.4, weight='weight')
node_sizes = [300*draw_G.degree(n)/max(dict(draw_G.degree()).values())+50 for n in draw_G.nodes()]
node_colors = [degree_cent.get(n,0) for n in draw_G.nodes()]
edge_widths = [min(draw_G[u][v].get('weight',1)*0.5, 4) for u,v in draw_G.edges()]
fig, ax = plt.subplots(figsize=(14,10))
nx.draw_networkx_edges(draw_G, pos, ax=ax, alpha=0.25, width=edge_widths, edge_color='#888888')
sc = nx.draw_networkx_nodes(draw_G, pos, ax=ax, node_size=node_sizes, node_color=node_colors, cmap='YlOrRd', alpha=0.9)
label_nodes = {n:n for n,_ in sorted(draw_G.degree(), key=lambda x:x[1], reverse=True)[:20]}
nx.draw_networkx_labels(draw_G, pos, labels=label_nodes, font_size=7, ax=ax)
plt.colorbar(sc, ax=ax, label='Degree Centrality')
ax.set_title('Author Co-Activity Network', fontsize=13); ax.axis('off'); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'4_author_network.png'), dpi=150); plt.show(); plt.close()

try:
    from pyvis.network import Network as PyvisNet
    print('Generating interactive HTML network \u2026')
    net = PyvisNet(height='800px', width='100%', bgcolor='#1a1a2e', font_color='white', notebook=False)
    net.barnes_hut(gravity=-8000, central_gravity=0.3, spring_length=150, damping=0.09)
    max_deg = max(dict(draw_G.degree()).values()) or 1; cmap_fn = cm.get_cmap('YlOrRd')
    for node in draw_G.nodes():
        dc = degree_cent.get(node,0); rgba = cmap_fn(dc)
        color = '#{:02x}{:02x}{:02x}'.format(int(rgba[0]*255), int(rgba[1]*255), int(rgba[2]*255))
        size = 10+40*(draw_G.degree(node)/max_deg)
        title = f'<b>{node}</b><br>Degree: {dc:.4f}<br>Betweenness: {betw_cent.get(node,0):.4f}'
        net.add_node(node, label=node, color=color, size=size, title=title)
    for u,v,data in draw_G.edges(data=True):
        w = data.get('weight',1); net.add_edge(u, v, value=w, title=f'Shared threads: {w}', color='rgba(180,180,180,0.4)')
    html_path = os.path.join(OUTPUT_DIR,'5_interactive_network.html'); net.save_graph(html_path)
    print(f'Saved interactive network -> {html_path}')
except ImportError:
    print('pyvis not installed -- skipping interactive HTML')

print('\nBuilding thread similarity network \u2026')
top_t_set = set(thread_stats.head(80).index); df_t = df[df['thread_id'].isin(top_t_set)]
author_threads = defaultdict(set)
for _, row in df_t.iterrows(): author_threads[row['author']].add(row['thread_id'])
thread_edge = defaultdict(int)
for auth, threads in tqdm(author_threads.items(), desc='Thread edges'):
    for t1, t2 in itertools.combinations(sorted(threads), 2): thread_edge[(t1,t2)] += 1
T = nx.Graph(); T.add_nodes_from(top_t_set)
for (t1,t2), w in thread_edge.items():
    if w >= 2: T.add_edge(t1, t2, weight=w)
T.remove_nodes_from(list(nx.isolates(T)))
print(f'Thread network: {T.number_of_nodes()} nodes, {T.number_of_edges()} edges')

pos_t = nx.spring_layout(T, seed=42, weight='weight')
thread_size_map = thread_stats['comment_count'].to_dict()
node_sz = [max(30, thread_size_map.get(n,50))*0.3 for n in T.nodes()]
fig, ax = plt.subplots(figsize=(12,8))
nx.draw_networkx_edges(T, pos_t, ax=ax, alpha=0.3, edge_color='#aaaaaa')
nx.draw_networkx_nodes(T, pos_t, ax=ax, node_size=node_sz, node_color=node_sz, cmap='cool', alpha=0.85)
top_t_labels = {n:n[-8:] for n in sorted(T.nodes(), key=lambda x:thread_size_map.get(x,0), reverse=True)[:15]}
nx.draw_networkx_labels(T, pos_t, labels=top_t_labels, font_size=7, ax=ax)
ax.set_title('Thread Similarity Network\n(edges = shared authors >= 2)', fontsize=13); ax.axis('off'); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'6_thread_network.png'), dpi=150); plt.show(); plt.close()

print(f'\n\u2705 Network analysis complete! Outputs in: {OUTPUT_DIR}')
print(f'\n\u2705\u2705\u2705 ALL 3 ANALYSES FINISHED SUCCESSFULLY \u2705\u2705\u2705')

Loading data …


Reading: 369292it [00:04, 89816.30it/s]


Loaded 318,604 comments from 62,933 authors in 501 threads

Top 15 most active authors:
                     comment_count  total_score    mean_score  threads_active
author                                                                       
TannedCroissant                884      4990020   5644.819005             134
Poem_for_your_sprog            428      1569404   3666.831776              74
poopellar                      408      1404292   3441.892157              70
-eDgAR-                        328      1821216   5552.487805              68
elee0228                       288       985720   3422.638889              52
A_Very_Brave_Taco              280      6407336  22883.342857               1
insertstalem3me                224       718688   3208.428571              31
maleorderbride                 212      1252352   5907.320755              22
DeathSpiral321                 192      1530092   7969.229167              31
CockDaddyKaren                 188       319880   1701

Building edges: 100%|██████████| 192/192 [00:00<00:00, 128049.99it/s]


Raw edges: 1,693 -- filtering by min_coappear=3
Edges after filter: 58
Graph: 36 nodes, 58 edges
Computing centrality metrics …

Top 10 by Degree Centrality:
  TannedCroissant                 0.8000
  elee0228                        0.3714
  poopellar                       0.2857
  Poem_for_your_sprog             0.2286
  -eDgAR-                         0.2000
  insertstalem3me                 0.1143
  Portarossa                      0.1143
  Aminar14                        0.0857
  PM-Me-Your-TitsPlz              0.0857
  DeathSpiral321                  0.0857

Top 10 by Betweenness Centrality:
  TannedCroissant                 0.7464
  poopellar                       0.1746
  elee0228                        0.1683
  -eDgAR-                         0.1032
  DeathSpiral321                  0.0571
  Poem_for_your_sprog             0.0189
  PM-Me-Your-TitsPlz              0.0036
  XxsquirrelxX                    0.0000
  maleorderbride                  0.0000
  pjabrony                  

Thread edges: 100%|██████████| 11919/11919 [00:00<00:00, 1094873.18it/s]


Thread network: 60 nodes, 127 edges

✅ Network analysis complete! Outputs in: /content/drive/MyDrive/outputs2/network

✅✅✅ ALL 3 ANALYSES FINISHED SUCCESSFULLY ✅✅✅


In [ ]:
#@title 2. Install Dependencies
!pip install -q vaderSentiment pandas matplotlib seaborn scipy tqdm scikit-learn networkx pyvis sentence-transformers transformers torch


In [ ]:
# --- Part 4 — Advanced Sentiment with Transformer Embeddings ---
import json, os, warnings, numpy as np, pandas as pd, torch
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, seaborn as sns
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

warnings.filterwarnings('ignore')

# Setup Output
OUTPUT_DIR = os.path.join(BASE_OUT, 'embedding_analysis')
os.makedirs(OUTPUT_DIR, exist_ok=True)
SAMPLE_SIZE = 5000; BATCH_SIZE = 32; RANDOM_STATE = 42
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Loading data (sample={SAMPLE_SIZE}) …')
records = []
with open(DATA_PATH, 'r', encoding='utf-8') as fh:
    for i, line in enumerate(tqdm(fh, desc='Reading')):
        try: records.append(json.loads(line))
        except json.JSONDecodeError: continue
        if SAMPLE_SIZE and len(records) >= SAMPLE_SIZE: break
df_emb = pd.DataFrame(records)
df_emb['score'] = pd.to_numeric(df_emb['score'], errors='coerce').fillna(0)
df_emb['body'] = df_emb['body'].astype(str)

print('\nGenerating Transformer Embeddings (all-MiniLM-L6-v2) …')
embed_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
embeddings = embed_model.encode(df_emb['body'].tolist(), batch_size=BATCH_SIZE, show_progress_bar=True)

print('\nRunning DistilBERT Sentiment Analysis …')
sentiment_pipeline = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english', device=0 if device=='cuda' else -1)
results = []
for i in tqdm(range(0, len(df_emb), BATCH_SIZE), desc='Sentiment'):
    batch_texts = [t[:1000] for t in df_emb['body'].iloc[i : i+BATCH_SIZE].tolist()]
    results.extend(sentiment_pipeline(batch_texts))
df_emb['bert_sentiment'] = [r['label'] for r in results]
df_emb['bert_score'] = [r['score'] for r in results]

print('\nReducing dimensions (PCA -> t-SNE) …')
pca = PCA(n_components=min(50, len(df_emb)))
X_pca = pca.fit_transform(embeddings)
tsne = TSNE(n_components=2, perplexity=30, random_state=RANDOM_STATE, init='pca', learning_rate='auto')
X_tsne = tsne.fit_transform(X_pca)
df_emb['tsne_1'], df_emb['tsne_2'] = X_tsne[:, 0], X_tsne[:, 1]

print('\nIdentifying semantic clusters …')
kmeans = KMeans(n_clusters=5, random_state=RANDOM_STATE)
df_emb['semantic_cluster'] = kmeans.fit_predict(embeddings)

print('\nGenerating Plots …')
# 1. Sentiment Distribution
plt.figure(figsize=(7,7))
counts = df_emb['bert_sentiment'].value_counts()
plt.pie(counts, labels=counts.index, autopct='%1.1f%%', colors=['#e74c3c','#2ecc71'], startangle=140)
plt.title('BERT Sentiment Distribution'); plt.savefig(os.path.join(OUTPUT_DIR, '1_sentiment_pie.png'), dpi=150); plt.show(); plt.close()

# 2. t-SNE Map
plt.figure(figsize=(10,8))
sns.scatterplot(x='tsne_1', y='tsne_2', hue='bert_sentiment', data=df_emb, palette={'POSITIVE':'#2ecc71','NEGATIVE':'#e74c3c'}, alpha=0.6, s=15)
plt.title('t-SNE Projection (Colored by BERT Sentiment)'); plt.savefig(os.path.join(OUTPUT_DIR, '2_tsne_sentiment.png'), dpi=150); plt.show(); plt.close()

# 3. Heatmap
plt.figure(figsize=(10,6))
ct = pd.crosstab(df_emb['semantic_cluster'], df_emb['bert_sentiment'], normalize='index')*100
sns.heatmap(ct, annot=True, fmt='.1f', cmap='YlGnBu')
plt.title('Sentiment % per Semantic Cluster'); plt.savefig(os.path.join(OUTPUT_DIR, '3_cluster_sentiment_heatmap.png'), dpi=150); plt.show(); plt.close()

out_csv = os.path.join(OUTPUT_DIR, 'bert_sentiment_results.csv')
df_emb.to_csv(out_csv, index=False)
print(f'\n✅ Advanced analysis complete! Outputs in: {OUTPUT_DIR}')


Loading data (sample=5000) …


Reading: 4999it [00:00, 76174.19it/s]



Generating Transformer Embeddings (all-MiniLM-L6-v2) …


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Sentiment: 100%|██████████| 157/157 [01:03<00:00,  2.47it/s]



Reducing dimensions (PCA -> t-SNE) …

Identifying semantic clusters …

Generating Plots …

✅ Advanced analysis complete! Outputs in: /content/drive/MyDrive/outputs2/embedding_analysis


In [ ]:
#@title 4. Copy outputs to Google Drive (optional)
import shutil
drive_out = '/content/drive/MyDrive/dva/outputs'
if os.path.exists('/content/drive/MyDrive'):
    shutil.copytree(BASE_OUT, drive_out, dirs_exist_ok=True)
    print(f'\u2705 All outputs copied to {drive_out}')
else:
    print('Drive not mounted, outputs are in /content/outputs/')